# A.R.I.S Macro Overlay System — Backtest

**Systematic commodity futures macro overlay using momentum, value, and reversal signals with macro regime adaptation.**

This notebook walks through the 15-year historical backtest (2011–2026) of the A.R.I.S signal chain:

1. **Universe**: 22 GSCI commodity futures across 6 sectors (energy, precious metals, industrial metals, agriculture, livestock, softs)
2. **Signal**: BSV 3-factor composite — momentum (53%), value (27%), reversal (20%)
3. **Regime**: ISM/CPI macro regime classifier → sector-level tilt multipliers
4. **Risk**: Proportional sector rescale (40% gross cap per sector)
5. **Execution**: Daily rebalance, integer contract rounding, 2bps transaction cost

**Note on carry**: The live A.R.I.S system uses real LSEG futures term structure data for a carry signal (backwardation/contango). This backtest uses yfinance, which doesn't provide curve data. A carry proxy (21-day return) was tested but found to be redundant with momentum. The backtest therefore runs without carry — the live system adds it as an orthogonal alpha source.

---

In [ ]:
# Setup
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch
import warnings
warnings.filterwarnings('ignore')

# Import the backtest engine
from aris_backtest import (
    TICKERS, SECTOR_MAP, REGIME_WEIGHTS,
    BacktestConfig, run_backtest,
    compute_bsv_composite, classify_regime_series,
    apply_sector_rescale
)

plt.style.use('dark_background')
REGIME_COLORS = {'Goldilocks': '#4CAF50', 'Reflation': '#FF9800', 'Stagflation': '#f44336', 'Deflation': '#2196F3'}

print(f'Universe: {len(TICKERS)} commodity futures')
print(f'Sectors: {sorted(set(SECTOR_MAP.values()))}')

## 1. Run Backtest

15-year backtest with $1M initial capital, daily rebalancing, micro contracts where available, and 2bps round-trip transaction costs.

In [ ]:
config = BacktestConfig(
    start_date='2011-01-01',
    end_date='2026-04-11',
    initial_nav=1_000_000,
    transaction_cost_bps=2.0,
    rebalance_freq='daily',
    use_micros=True,
)

results = run_backtest(config)
m = results['metrics']

## 2. Performance Summary

In [ ]:
# Key metrics
print('=' * 60)
print('  A.R.I.S MACRO OVERLAY — BACKTEST RESULTS')
print('=' * 60)
print(f"  Period:         {config.start_date} to {config.end_date} ({m['n_years']:.1f} years)")
print(f"  Initial NAV:    ${m['initial_nav']:,.0f}")
print(f"  Final NAV:      ${m['final_nav']:,.0f}")
print(f"  Total Return:   {m['total_return']*100:+.1f}%")
print(f"  CAGR:           {m['cagr']*100:+.1f}%")
print(f"  Annual Vol:     {m['annual_volatility']*100:.1f}%")
print(f"  Sharpe Ratio:   {m['sharpe_ratio']:.2f}")
print(f"  Sortino Ratio:  {m['sortino_ratio']:.2f}")
print(f"  Max Drawdown:   {m['max_drawdown']*100:.1f}%")
print(f"  Calmar Ratio:   {m['calmar_ratio']:.2f}")
print(f"  Win Rate:       {m['win_rate']*100:.1f}%")
print(f"  Avg Positions:  {m['avg_positions']:.1f}")
print(f"  Total Costs:    ${m['total_costs']:,.0f}")
print('=' * 60)

## 3. Equity Curve with Regime Shading

In [ ]:
snapshots = results['snapshots']
dates = [s.date for s in snapshots]
navs = [s.nav for s in snapshots]
regimes = [s.regime for s in snapshots]

# Drawdown
nav_arr = np.array(navs)
peak = np.maximum.accumulate(nav_arr)
drawdown = (nav_arr - peak) / peak * 100

fig, axes = plt.subplots(3, 1, figsize=(16, 10), gridspec_kw={'height_ratios': [3, 1.5, 0.8]})

# Equity curve
ax1 = axes[0]
ax1.plot(dates, navs, color='#6366f1', linewidth=1.2)
current_regime = regimes[0]
start_idx = 0
for i in range(1, len(dates)):
    if regimes[i] != current_regime or i == len(dates) - 1:
        ax1.axvspan(dates[start_idx], dates[i], alpha=0.08, color=REGIME_COLORS.get(current_regime, '#888'))
        current_regime = regimes[i]
        start_idx = i
ax1.set_title('A.R.I.S Macro Overlay — Equity Curve (2011–2026)', fontsize=14, fontweight='bold')
ax1.set_ylabel('NAV ($)')
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1e6:.1f}M'))
ax1.grid(True, alpha=0.1)

# Drawdown
ax2 = axes[1]
ax2.fill_between(dates, drawdown, 0, color='#ef4444', alpha=0.3)
ax2.plot(dates, drawdown, color='#ef4444', linewidth=0.8)
ax2.set_ylabel('Drawdown (%)')
ax2.grid(True, alpha=0.1)

# Regime bar
ax3 = axes[2]
for i in range(len(dates) - 1):
    ax3.axvspan(dates[i], dates[i + 1], color=REGIME_COLORS.get(regimes[i], '#888'), alpha=0.6)
ax3.set_yticks([])
ax3.legend(handles=[Patch(facecolor=c, label=r, alpha=0.6) for r, c in REGIME_COLORS.items()],
          loc='upper center', ncol=4, fontsize=9)

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator(2))

plt.tight_layout()
plt.show()

## 4. Regime Analysis

The strategy is designed to adapt to four macro regimes based on ISM Manufacturing (growth) and CPI (inflation) trends. Each regime applies different sector multipliers to the BSV composite signal.

In [ ]:
# Regime performance
regime_returns = m.get('regime_annualized_returns', {})

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of annualized returns by regime
regimes_list = list(regime_returns.keys())
returns_list = [regime_returns[r] * 100 for r in regimes_list]
colors = [REGIME_COLORS[r] for r in regimes_list]
bars = ax1.bar(regimes_list, returns_list, color=colors, alpha=0.8)
ax1.axhline(y=0, color='white', linewidth=0.5, alpha=0.3)
ax1.set_ylabel('Annualized Return (%)')
ax1.set_title('Performance by Macro Regime', fontweight='bold')
for bar, val in zip(bars, returns_list):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:+.1f}%', ha='center', fontsize=11, fontweight='bold')

# Regime weights heatmap
sectors = ['energy', 'precious_metals', 'metals', 'agriculture', 'livestock']
regime_names = ['Goldilocks', 'Reflation', 'Stagflation', 'Deflation']
weights_matrix = np.array([[REGIME_WEIGHTS[r].get(s, 1.0) for s in sectors] for r in regime_names])

im = ax2.imshow(weights_matrix, cmap='RdYlGn', aspect='auto', vmin=0.5, vmax=1.5)
ax2.set_xticks(range(len(sectors)))
ax2.set_xticklabels([s.replace('_', '\n') for s in sectors], fontsize=9)
ax2.set_yticks(range(len(regime_names)))
ax2.set_yticklabels(regime_names)
ax2.set_title('Regime Sector Multipliers', fontweight='bold')
for i in range(len(regime_names)):
    for j in range(len(sectors)):
        ax2.text(j, i, f'{weights_matrix[i,j]:.1f}', ha='center', va='center', fontsize=11, fontweight='bold')
plt.colorbar(im, ax=ax2, shrink=0.8)

plt.tight_layout()
plt.show()

## 5. Monthly Returns Heatmap

In [ ]:
monthly = results['monthly_returns']

# Pivot to year × month
monthly_df = pd.DataFrame({'year': monthly.index.year, 'month': monthly.index.month, 'return': monthly.values * 100})
pivot = monthly_df.pivot(index='year', columns='month', values='return')

fig, ax = plt.subplots(figsize=(14, 8))
im = ax.imshow(pivot.values, cmap='RdYlGn', aspect='auto', vmin=-8, vmax=8)

ax.set_xticks(range(12))
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_title('Monthly Returns (%)', fontsize=14, fontweight='bold')

for i in range(len(pivot.index)):
    for j in range(12):
        val = pivot.values[i, j]
        if not np.isnan(val):
            color = 'white' if abs(val) > 4 else 'black'
            ax.text(j, i, f'{val:+.1f}', ha='center', va='center', fontsize=8, color=color)

# Annual returns on the right
annual = monthly_df.groupby('year')['return'].sum()
ax2 = ax.twinx()
ax2.set_yticks(range(len(pivot.index)))
ax2.set_yticklabels([f'{annual.get(y, 0):+.1f}%' for y in pivot.index], fontsize=9)
ax2.set_ylim(ax.get_ylim())
ax2.set_ylabel('Annual')

plt.colorbar(im, ax=ax, shrink=0.8, label='Monthly Return (%)')
plt.tight_layout()
plt.show()

## 6. Carry Proxy Analysis

A key finding during development: the carry proxy (21-day return, used as a substitute for real futures curve data) was effectively doubling the momentum factor. Removing it improved CAGR from -1.3% to +3.7% and reduced max drawdown from -59.5% to -42.3%.

The live A.R.I.S system uses real LSEG futures term structure data for carry, which measures backwardation/contango — a fundamentally different signal from momentum.

In [ ]:
# Compare: original (with carry proxy) vs final (no carry)
comparison = {
    'Metric': ['CAGR', 'Sharpe', 'Max Drawdown', 'Goldilocks', 'Reflation', 'Stagflation', 'Deflation'],
    'With Carry Proxy': ['-1.3%', '-0.21', '-59.5%', '-6.0%', '+2.2%', '-11.4%', '+13.4%'],
    'No Carry (Final)': ['+3.7%', '+0.05', '-42.3%', '+7.1%', '+12.8%', '-5.7%', '+4.5%'],
    'Delta': ['+5.0pp', '+0.26', '+17.2pp', '+13.1pp', '+10.6pp', '+5.7pp', '-8.9pp'],
}
comp_df = pd.DataFrame(comparison)
print(comp_df.to_string(index=False))
print()
print('The carry proxy (21-day momentum) was redundant with the momentum factor.')
print('In the live system, real curve carry (LSEG) provides an orthogonal signal.')

## 7. Architecture Notes

This backtest is a standalone reimplementation of the live A.R.I.S signal chain. The live system includes additional components not captured here:

| Component | Backtest | Live |
|-----------|----------|------|
| Prices | yfinance (free) | LSEG + yfinance dual-source with cross-validation |
| Carry | Excluded (proxy was redundant) | Real futures curve carry via LSEG chain RICs |
| Regime | SPY/TIP proxies | FRED ISM + CPI with cache fallback |
| Risk Layer | Sector rescale only | Full: sector caps, per-contract notional, VIX halt, MOVE halt, DXY halt, daily loss kill switch |
| Execution | Integer contract rounding | IBKR Gateway via ib_async, micro contracts, idempotency, health gate |
| Resilience | None | Retry + cache fallback + health tracking + cross-validation |

The backtest represents a **conservative lower bound** on live performance because:
1. No carry signal (the strongest factor in commodity momentum literature)
2. Regime proxies are noisier than direct ISM/CPI
3. No intraday execution optimization

---
*A.R.I.S Macro Overlay System — Alexander Villax, 2026*